# Module 1: Data Profiling & Quality

### Q1. Create a complete Data Quality Report showing data type, null count, null percentage, distinct count, duplicate count, and min/max values where applicable.

- Objective: 
Solve the following advanced-level Python Pandas and Data Analytics exercises using the Telecom Customer Churn dataset.

In [1]:
import pandas as pd

df = pd.read_csv("/Users/andreams/capstone_project/datasets/telecom_customers_churn.csv")

df.shape

(7043, 21)

In [2]:
# DATA QUALITY REPORT

data_quality_report = pd.DataFrame()

data_quality_report["data_type"] = df.dtypes
data_quality_report["null_count"] = df.isnull().sum()
data_quality_report["null_percentage"] = (df.isnull().mean() * 100).round(2)
data_quality_report["distinct_count"] = df.nunique()
data_quality_report["duplicate_count_rows"] = df.duplicated().sum()
data_quality_report["duplicate_value_count"] = df.apply(lambda x: x.duplicated().sum())

data_quality_report["min_value"] = df.select_dtypes(include="number").min()
data_quality_report["max_value"] = df.select_dtypes(include="number").max()

data_quality_report

,data_type,null_count,null_percentage,distinct_count,duplicate_count_rows,duplicate_value_count,min_value,max_value
customerID,object,0,0.0,7043,0,0,NaN,NaN
gender,object,0,0.0,2,0,7041,NaN,NaN
SeniorCitizen,int64,0,0.0,2,0,7041,0.00,1.00
Partner,object,0,0.0,2,0,7041,NaN,NaN
Dependents,object,0,0.0,2,0,7041,NaN,NaN
tenure,int64,0,0.0,73,0,6970,0.00,72.00
PhoneService,object,0,0.0,2,0,7041,NaN,NaN
MultipleLines,object,0,0.0,3,0,7040,NaN,NaN
InternetService,object,0,0.0,3,0,7040,NaN,NaN
OnlineSecurity,object,0,0.0,3,0,7040,NaN,NaN


### Q2. The TotalCharges column is stored as an object. Identify invalid values, convert it to numeric, and quantify affected records.

In [3]:
# Identify invalid values

## give me the original TotalCharges values where the conversion produced NaN, but the original value was NOT already NaN.
invalid_values = df["TotalCharges"][
    pd.to_numeric(df["TotalCharges"], errors="coerce").isna()
    & df["TotalCharges"].notna()
]

# quantify affected records 
invalid_count = invalid_values.count()

print(invalid_count)

11


In [4]:
# convert it to numeric

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

print(df["TotalCharges"].dtype)
print(df.isna().sum())

float64
customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64


### Q3. Create a function that automatically categorizes columns into Numerical, Categorical, Binary, and Identifier groups.

In [5]:
def categorize_columns(df):
    numerical = []
    categorical = []
    binary = []
    identifier = []

    for column in df.columns:
        if column.lower().endswith("id"):
            identifier.append(column)
        elif df[column].nunique() == 2:
            binary.append(column)
        elif pd.api.types.is_numeric_dtype(df[column]):
            numerical.append(column)
        else:
            categorical.append(column)
    return {
        "Numerical": numerical,
        "Categorical": categorical,
        "Binary": binary,
        "Identifier": identifier
    }

categories = categorize_columns(df)

print(categories)

{'Numerical': ['tenure', 'MonthlyCharges', 'TotalCharges'], 'Categorical': ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod'], 'Binary': ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn'], 'Identifier': ['customerID']}


### Q4. Identify customers where TotalCharges differs from MonthlyCharges × Tenure by more than 10%. Investigate possible causes.

In [6]:
# expected total charges 

df["expected_total_charges"] = df["MonthlyCharges"] * df["tenure"]

# percentage difference 

df["difference_percentage"] = (
    abs(df["TotalCharges"] - df["expected_total_charges"])
    / df["expected_total_charges"]
) * 100

# TotalCharges differs from MonthlyCharges × Tenure by more than 10%

affected_customers = df[df["difference_percentage"] > 10]

# affected customers

affected_customers[
    [
        "customerID",
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "expected_total_charges",
        "difference_percentage"

    ]
]

print("Number of affected customers:", len(affected_customers))

Number of affected customers: 381


In [7]:
### overview of comparison the characteristics of affected customers with all customers

possible_causes = []

for column in categories["Categorical"] + categories["Binary"]:
    affected_pct = affected_customers[column].value_counts(normalize=True) * 100
    all_pct = df[column].value_counts(normalize=True) * 100
    
    comparison = pd.DataFrame({
        "Affected %": affected_pct,
        "All %": all_pct
    }).fillna(0)
    
    comparison["Difference"] = (
        comparison["Affected %"] - comparison["All %"]
    )
    
    comparison["Column"] = column
    comparison["Category"] = comparison.index
    
    possible_causes.append(comparison)

possible_causes = pd.concat(possible_causes)

# show the biggest overrepresented categories
possible_causes.sort_values("Difference", ascending=False).head(10)

,Affected %,All %,Difference,Column,Category
No internet service,50.131234,21.666903,28.464330,StreamingMovies,No internet service
No internet service,50.131234,21.666903,28.464330,OnlineBackup,No internet service
No internet service,50.131234,21.666903,28.464330,StreamingTV,No internet service
No internet service,50.131234,21.666903,28.464330,TechSupport,No internet service
No,50.131234,21.666903,28.464330,InternetService,No
No internet service,50.131234,21.666903,28.464330,DeviceProtection,No internet service
No internet service,50.131234,21.666903,28.464330,OnlineSecurity,No internet service
Mailed check,44.881890,22.887974,21.993916,PaymentMethod,Mailed check
No,69.816273,48.132898,21.683375,MultipleLines,No
Month-to-month,71.391076,55.019168,16.371908,Contract,Month-to-month


In [8]:
# true → customers whose difference_percentage is greater than 10% → your affected customers
# false → customers whose difference is 10% or less → non-affected customers

possible_causes_numerical = df.groupby(
    df["difference_percentage"] > 10
)[["tenure", "MonthlyCharges", "TotalCharges"]].agg(["mean", "median"])

possible_causes_numerical

tenure        MonthlyCharges         TotalCharges  \
                            mean median           mean  median         mean   
difference_percentage                                                         
False                  33.665416   31.0      66.213389  71.925  2396.583288   
True                    9.740157    6.0      39.377953  25.100   305.756562   

                                
                        median  
difference_percentage           
False                  1532.45  
True                    218.55

### Possible causes: 

- Customers with no internet service, mailed-check payment methods, no multiple lines, and month-to-month contracts are overrepresented among customers whose TotalCharges differs from MonthlyCharges × tenure by more than 10%. These characteristics may therefore be associated with the discrepancy.

- The customers with >10% discrepancies are heavily concentrated among customers with short tenure.

- The affected customers are characterized by shorter tenure and are more likely to have no internet service, use mailed checks, and have month-to-month contracts compared with the overall customer population. These characteristics may be associated with the discrepancy between TotalCharges and MonthlyCharges × tenure.

# Module 2: Advanced Customer Analytics

Formula churn rate: 

Churn Rate = Number of customers who churned / Total number of customers * 100

### Q6. Calculate churn rate by Gender, Senior Citizen, Partner, and Dependents. Rank the highest-risk customer groups.

In [43]:
### churn_rate_gender

churn_rate_gender = (
    df.groupby("gender")["Churn"]
      .apply(lambda x: (x == "Yes").mean() * 100)
)

### churn_rate_senior

churn_rate_senior = (
    df.groupby("SeniorCitizen")["Churn"]
      .apply(lambda x: (x == "Yes").mean() * 100)
      .rename(index={1: "Yes", 0: "No"})
)

### churn_rate_partner

churn_rate_partner = (
    df.groupby("Partner")["Churn"]
      .apply(lambda x: (x == "Yes").mean() * 100)
)
### churn_rate_dependents

churn_rate_dependents = (
    df.groupby("Dependents")["Churn"]
        .apply(lambda x: (x == "Yes").mean() * 100)
)

print("churn_rate_gender:")
print(churn_rate_gender)

print("\nchurn_rate_senior:")
print(churn_rate_senior)

print("\nchurn_rate_partner:")
print(churn_rate_partner)

print("\nchurn_rate_dependents:")
print(churn_rate_dependents)

churn_rate_gender:
gender
Female    26.920872
Male      26.160338
Name: Churn, dtype: float64

churn_rate_senior:
SeniorCitizen
No     23.606168
Yes    41.681261
Name: Churn, dtype: float64

churn_rate_partner:
Partner
No     32.957979
Yes    19.664903
Name: Churn, dtype: float64

churn_rate_dependents:
Dependents
No     31.279140
Yes    15.450237
Name: Churn, dtype: float64


In [45]:
## Rank the highest-risk customer groups.

ranking_higher_risk_customer_groups = pd.concat([
    churn_rate_gender,
    churn_rate_senior,
    churn_rate_partner,
    churn_rate_dependents
], keys=["Gender", "Senior Citizen", "Partner", "Dependents"])

ranking_higher_risk_customer_groups = ranking_higher_risk_customer_groups.sort_values(ascending=False)

print(ranking_higher_risk_customer_groups)

Senior Citizen  Yes       41.681261
Partner         No        32.957979
Dependents      No        31.279140
Gender          Female    26.920872
                Male      26.160338
Senior Citizen  No        23.606168
Partner         Yes       19.664903
Dependents      Yes       15.450237
Name: Churn, dtype: float64


Senior citizens are the highest-risk customer group, with a churn rate of 41.7%. Customers without a partner (33.0%) and without dependents (31.3%) also have relatively high churn rates. Gender appears to have little impact on churn, as the rates for females (26.9%) and males (26.2%) are very similar. Customers with dependents have the lowest churn rate (15.5%).

### Q7. Create customer segments based on tenure and analyze churn across segments.

In [46]:
### customer segments based on tenure

def create_customer_segment_tenure(tenure):
    if tenure <=12:
        return "New"
        
    elif tenure <= 24:
        return "Early"

    elif tenure <= 48:
        return "Established"
        
    else:
        return "Long-term"

df["customer_segment_tenure"] = df["tenure"].apply(create_customer_segment_tenure)

### analyze churn across segments

churn_segment = (
    df.groupby("customer_segment_tenure")["Churn"]
      .apply(lambda x: (x == "Yes").mean() * 100)
      .sort_values(ascending=False)
)

print(churn_segment)

customer_segment_tenure
New            47.438243
Early          28.710938
Established    20.388959
Long-term       9.513176
Name: Churn, dtype: float64


New customers are more likely to churn

### Q8. Identify the top 10 customer profiles most likely to churn using combinations of Contract, Payment Method, and Internet Service.

In [49]:
top_10_customers_profiles_to_churn = (
    df.groupby(["Contract", "PaymentMethod", "InternetService"])["Churn"]
    .agg(
        customers="size",
        churned=lambda x: (x == "Yes").sum()
    )
    .reset_index()
)

top_10_customers_profiles_to_churn["churn_rate"] = (
    top_10_customers_profiles_to_churn["churned"]
    / top_10_customers_profiles_to_churn["customers"]
    * 100
)

top_10_customers_profiles_to_churn = (
    top_10_customers_profiles_to_churn
    .sort_values("churn_rate", ascending=False)
    .head(10)
)


print(top_10_customers_profiles_to_churn)

          Contract              PaymentMethod InternetService  customers  \
7   Month-to-month           Electronic check     Fiber optic       1307   
10  Month-to-month               Mailed check     Fiber optic        201   
1   Month-to-month  Bank transfer (automatic)     Fiber optic        327   
4   Month-to-month    Credit card (automatic)     Fiber optic        293   
6   Month-to-month           Electronic check             DSL        474   
9   Month-to-month               Mailed check             DSL        367   
3   Month-to-month    Credit card (automatic)             DSL        185   
19        One year           Electronic check     Fiber optic        196   
11  Month-to-month               Mailed check              No        325   
2   Month-to-month  Bank transfer (automatic)              No         65   

    churned  churn_rate  
7       789   60.367253  
10      102   50.746269  
1       149   45.565749  
4       122   41.638225  
6       192   40.506329  
9      

The analysis shows that month-to-month customers have the highest churn across most customer profiles. The highest-churn profile is month-to-month customers using electronic checks and fiber optic internet, with a churn rate of 60.4% (789 out of 1,307 customers). Overall, month-to-month contracts, electronic check payment, and fiber optic internet are frequently associated with high-churn customer profiles.

### Q9. Calculate Customer Lifetime Value (CLV = MonthlyCharges × Tenure) and compare churned vs retained customers.

In [59]:
# customer lifetime value calculation

df["CLV"] = df["MonthlyCharges"] * df["tenure"]


## churned vs retained customers
clv_churn_vs_retained = (
    df.groupby("Churn")
    .agg(
        customers=("CLV", "size"),
        average_clv=("CLV", "mean")
    )
    .reset_index()
)

print(clv_churn_vs_retained.round(2))

  Churn  customers  average_clv
0    No       5174      2549.77
1   Yes       1869      1531.61


### Q10. Create a Customer Risk Score using Contract Type, Tenure, Tech Support, and Online Security.

In [62]:
## check churn rate OnlineSecurity
df.groupby("OnlineSecurity")["Churn"].apply(
    lambda x: (x == "Yes").mean() * 100
)

OnlineSecurity
No                     41.766724
No internet service     7.404980
Yes                    14.611194
Name: Churn, dtype: float64

In [63]:
## check churn rate TechSupport
df.groupby("TechSupport")["Churn"].apply(
    lambda x: (x == "Yes").mean() * 100
)

TechSupport
No                     41.635474
No internet service     7.404980
Yes                    15.166341
Name: Churn, dtype: float64

In [65]:
## check churn rate based on contract 

df.groupby("Contract")["Churn"].apply(
    lambda x: (x == "Yes").mean() * 100
)

Contract
Month-to-month    42.709677
One year          11.269518
Two year           2.831858
Name: Churn, dtype: float64

In [70]:
df["customer_risk_score"] = (
    (df["Contract"] == "Month-to-month").astype(int)
    + (df["tenure"] < 12).astype(int)
    + (df["TechSupport"] == "No").astype(int)
    + (df["OnlineSecurity"] == "No").astype(int)
)

print(df[[
    "Contract",
    "tenure", 
    "TechSupport", 
    "OnlineSecurity", 
    "customer_risk_score"
]].head(10))

         Contract  tenure TechSupport OnlineSecurity  customer_risk_score
0  Month-to-month       1          No             No                    4
1        One year      34          No            Yes                    1
2  Month-to-month       2          No            Yes                    3
3        One year      45         Yes            Yes                    0
4  Month-to-month       2          No             No                    4
5  Month-to-month       8          No             No                    4
6  Month-to-month      22          No             No                    3
7  Month-to-month      10          No            Yes                    3
8  Month-to-month      28         Yes             No                    2
9        One year      62          No            Yes                    1
